In [0]:
# Databricks notebook source
# MAGIC 

In [0]:
%run ./shared_audit_utils

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import DataFrame

import logging
from datetime import datetime

DataFrame[]

In [0]:
# Get run id
run_id = get_run_id()
start_time = datetime.now()

In [0]:
# Logger
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

logger = logging.getLogger(
    "gold_pipeline"
)

In [0]:
# Configuration
dbutils.widgets.text(
    "pipeline_name",
    "online_retail_gold"
)

PIPELINE_NAME = dbutils.widgets.get(
    "pipeline_name"
)

logger.info(
    f"Pipeline Name: {PIPELINE_NAME}"
)

configs = (
    spark.table("workspace.default.pipeline_config")
    .filter(
        f"""
        pipeline_name = '{PIPELINE_NAME}'
        AND is_active = true
        """
    )
    .collect()
)

if len(configs) == 0:
    raise Exception(
        f"No metadata found for {PIPELINE_NAME}"
    )

SILVER_TABLE = configs[0].source_name

OUTPUT_TABLES = {
    row.output_type: row.target_table
    for row in configs
}

logger.info(
    f"Source Table: {SILVER_TABLE}"
)

logger.info(
    f"Output Tables: {OUTPUT_TABLES}"
)

2026-06-06 21:24:49,440 INFO Received command c on object id p0
2026-06-06 21:24:50,079 INFO Source Table: workspace.default.silver_online_retail
2026-06-06 21:24:50,081 INFO Output Tables: {'daily': 'workspace.default.gold_daily_sales', 'country': 'workspace.default.gold_country_sales', 'customer': 'workspace.default.gold_customer_sales'}


In [0]:
# Read Silver
def read_silver() -> DataFrame:
    logger.info(f"Reading {SILVER_TABLE}")
    return spark.table(
        SILVER_TABLE
    )

In [0]:
# Validate source
def validate_source(df: DataFrame):

    logger.info("Validating source")
    if df.limit(1).count() == 0:
        raise Exception("Silver table is empty")

In [0]:
# Daily Sales
def create_daily_sales(df: DataFrame) -> DataFrame:

    logger.info("Creating daily sales")

    return (
        df
        .groupBy(to_date(col("InvoiceDate")).alias("sales_date"))
        .agg(countDistinct("InvoiceNo").alias("total_orders"),
            round(sum("SalesAmount"),2).alias("total_revenue"),
            sum("Quantity").alias("total_quantity")
        )
    )

In [0]:
# Country Sales
def create_country_sales(df: DataFrame) -> DataFrame:

    logger.info("Creating country sales")

    return (
        df.groupBy("Country")
        .agg(countDistinct("InvoiceNo").alias("total_orders"),
            round(sum("SalesAmount"),2).alias("total_revenue"),
            sum("Quantity").alias("total_quantity")
        )
    )

In [0]:
# Customer Sales
def create_customer_sales(df: DataFrame) -> DataFrame:

    logger.info("Creating customer sales")

    return (
        df.filter(col("CustomerID").isNotNull())
        .groupBy("CustomerID")
        .agg(countDistinct("InvoiceNo").alias("total_orders"),
            round(sum("SalesAmount"),2).alias("total_revenue"),
            sum("Quantity").alias("total_quantity")
        )
    )

In [0]:
# Validate functions
def validate_daily_sales(df: DataFrame):

    null_dates = (
        df.filter(col("sales_date").isNull()).count()
    )

    logger.info(f"Null sales dates: {null_dates}")

    if null_dates > 0:
        raise Exception(
            "Gold daily sales contains null dates"
        )

In [0]:
# Audit metrics
def log_metrics(input_count,daily_count,country_count,customer_count):

    logger.info(f"Input Count: {input_count}")
    logger.info(f"Daily Sales Rows: {daily_count}")
    logger.info(f"Country Sales Rows: {country_count}")
    logger.info(f"Customer Sales Rows: {customer_count}")

In [0]:
# Write Functions
def write_gold_table(df: DataFrame,table_name: str):

    logger.info(f"Writing {table_name}")
    (
        df.write.format("delta").mode("overwrite").saveAsTable(table_name)
    )

In [0]:
# Main
def main(run_id,start_time):

    logger.info("Starting gold pipeline")

    silver_df = read_silver()

    validate_source(silver_df)

    input_count = (silver_df.count())
    
    daily_df = (create_daily_sales(silver_df))

    country_df = (create_country_sales(silver_df))

    customer_df = (create_customer_sales(silver_df))

    daily_count = (daily_df.count())

    country_count = (country_df.count())

    customer_count = (customer_df.count())

    log_metrics(
        input_count,
        daily_count,
        country_count,
        customer_count
    )
    validate_daily_sales(daily_df)

    write_gold_table(daily_df,OUTPUT_TABLES["daily"])

    write_gold_table(country_df,OUTPUT_TABLES["country"])

    write_gold_table(customer_df,OUTPUT_TABLES["customer"])

    logger.info("Gold pipeline completed")

    end_time = datetime.now()
    # Write audit record
    # logger.info("Writing logs to audit table")
    write_audit_record(
        run_id=run_id,
        pipeline_name="gold_daily_sales" ,
        start_time=start_time,
        end_time=end_time,
        status="SUCCESS",
        input_count=input_count,
        output_count=daily_count,
        error_message=None
    )

    write_audit_record(
        run_id=run_id,
        pipeline_name="gold_country_sales" ,
        start_time=start_time,
        end_time=end_time,
        status="SUCCESS",
        input_count=input_count,
        output_count=country_count,
        error_message=None
    )

    write_audit_record(
        run_id=run_id,
        pipeline_name="gold_customer_sales" ,
        start_time=start_time,
        end_time=end_time,
        status="SUCCESS",
        input_count=input_count,
        output_count=customer_count,
        error_message=None
    )


In [0]:
# Entry Point
if __name__ == "__main__":

    run_id = get_run_id()
    start_time = datetime.now()

    try:

        main(
            run_id,
            start_time
        )

    except Exception as e:

        end_time = datetime.now()

        write_audit_record(
            run_id=run_id,
            pipeline_name=PIPELINE_NAME,
            start_time=start_time,
            end_time=end_time,
            status="FAILED",
            input_count=0,
            output_count=0,
            error_message=str(e)
        )

        logger.exception(
        f"Pipeline {PIPELINE_NAME} failed: {e}"
        )

        raise

2026-06-06 21:25:01,205 INFO Starting gold pipeline
2026-06-06 21:25:01,206 INFO Reading workspace.default.silver_online_retail
2026-06-06 21:25:01,206 INFO Validating source
2026-06-06 21:25:02,176 INFO Creating daily sales
2026-06-06 21:25:02,208 INFO Creating country sales
2026-06-06 21:25:02,209 INFO Creating customer sales
2026-06-06 21:25:03,515 INFO Input Count: 530693
2026-06-06 21:25:03,516 INFO Daily Sales Rows: 305
2026-06-06 21:25:03,516 INFO Country Sales Rows: 38
2026-06-06 21:25:03,516 INFO Customer Sales Rows: 4339
2026-06-06 21:25:04,284 INFO Null sales dates: 0
2026-06-06 21:25:04,284 INFO Writing workspace.default.gold_daily_sales
2026-06-06 21:25:07,650 INFO Writing workspace.default.gold_country_sales
2026-06-06 21:25:10,554 INFO Writing workspace.default.gold_customer_sales
2026-06-06 21:25:13,409 INFO Gold pipeline completed
2026-06-06 21:25:13,440 INFO Writing logs to audit table
2026-06-06 21:25:15,221 INFO Writing logs to audit table
2026-06-06 21:25:17,270 IN

In [0]:
display(
    spark.table(
        "workspace.default.gold_daily_sales"
    )
)

sales_date,total_orders,total_revenue,total_quantity
2010-12-06,102,54830.46,21795
2011-02-16,63,24949.1,16156
2011-02-20,26,9624.69,5353
2011-03-11,54,25475.18,10822
2011-03-13,16,4148.12,2749
2011-03-22,53,31515.58,16568
2011-05-03,66,35229.41,12466
2011-05-19,99,35599.35,18144
2011-06-12,39,12517.06,9491
2011-06-13,59,22457.21,11302


In [0]:
display(
    spark.table(
        "workspace.default.gold_country_sales"
    )
    .orderBy(
        col("total_revenue").desc()
    )
)

Country,total_orders,total_revenue,total_quantity
United Kingdom,18194,9003097.96,4701272
Netherlands,95,285446.34,200937
EIRE,288,283453.96,147447
Germany,457,228867.14,119263
France,392,209715.11,112104
Australia,57,138521.31,84209
Spain,90,61577.11,27951
Switzerland,54,57089.9,30630
Belgium,98,41196.34,23237
Sweden,36,38378.33,36083


In [0]:
display(
    spark.table(
        "workspace.default.gold_customer_sales"
    )
    .orderBy(
        col("total_revenue").desc()
    )
)

CustomerID,total_orders,total_revenue,total_quantity
14646,74,280206.02,197491
18102,60,259657.3,64124
17450,46,194550.79,69993
16446,2,168472.5,80997
14911,201,143825.06,80515
12415,21,124914.53,77670
14156,55,117379.63,57885
17511,31,91062.38,64549
16029,63,81024.84,40208
12346,1,77183.6,74215
